In [1]:
import io
import zipfile
from pathlib import Path

import pandas as pd
import requests

In [2]:
_here = Path.cwd()
REPO_DIR = next((p for p in [_here, *_here.parents]
                 if (p / '.gitattributes').exists() or (p / '.git').exists()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open this notebook from inside the SM_Dissertation repository folder.")

PULLED_DIR = REPO_DIR / 'GEP Review Code' / 'Pulled Data'  

PULLED_DIR.mkdir(parents=True, exist_ok=True)
BOUNDARIES_ZIP  = PULLED_DIR / 'ne_50m_admin_0_countries.zip'
GEP_SUMMARY_CSV = PULLED_DIR / 'gep_summary.csv'

In [3]:
NE_URL = 'https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip'

r = requests.get(NE_URL, timeout=120)
r.raise_for_status()
BOUNDARIES_ZIP.write_bytes(r.content)
print(f"Saved boundaries to {BOUNDARIES_ZIP} ({len(r.content)/1e6:.1f} MB)")

Saved boundaries to c:\Users\USER\Documents\GitHub\SM_Dissertation\GEP Review Code\Pulled Data\ne_50m_admin_0_countries.zip (0.8 MB)


In [4]:
TARGET_SCENARIO = '0_0_0_0_0_0'

SSA_CODES = {
    'AGO':'ao', 'BEN':'bj', 'BWA':'bw', 'BFA':'bf', 'BDI':'bi',
    'CMR':'cm', 'CAF':'cf', 'TCD':'td', 'CPV':'cv', 'COD':'cd', 'COG':'cg',
    'COM':'km', 'CIV':'ci', 'DJI':'dj', 'GNQ':'gq', 'ERI':'er', 'ETH':'et',
    'GAB':'ga', 'GMB':'gm', 'GHA':'gh', 'GIN':'gn', 'GNB':'gw',
    'KEN':'ke', 'LSO':'ls', 'LBR':'lr', 'MDG':'mg', 'MWI':'mw',
    'MLI':'ml', 'MRT':'mr', 'MOZ':'mz', 'MUS':'mu', 'NAM':'na', 'NER':'ne',
    'NGA':'ng', 'RWA':'rw', 'SEN':'sn', 'SLE':'sl', 'SOM':'so', 'STP':'st',
    'SYC':'sc', 'ZAF':'za', 'SSD':'ss', 'SDN':'sd', 'SWZ':'sz', 'TZA':'tz',
    'TGO':'tg', 'UGA':'ug', 'ZMB':'zm', 'ZWE':'zw'
}

summary_results = []
for iso3, iso2 in SSA_CODES.items():
    url = f'https://gep-source-archive.s3.amazonaws.com/models-2021/{iso2}-2/outputs/{iso2}-2-scenarios-summaries.zip'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"{iso3}: HTTP {r.status_code}")
            continue
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            target_file = f'{iso2}-2-{TARGET_SCENARIO}_summary.csv'
            if target_file not in z.namelist():
                print(f"{iso3}: file not found — available: {z.namelist()[:3]}")
                continue
            with z.open(target_file) as f:
                df = pd.read_csv(f, index_col=0)

                # sum 2025 + 2030 increments for cumulative connections
                year_cols = [c for c in df.columns if c in ['2025', '2030']]

                def sum_row(row_name):
                    if row_name in df.index:
                        return df.loc[row_name, year_cols].sum()
                    return 0

                new_conn_grid = sum_row('2.New_Connections_Grid')
                new_conn_shs  = sum_row('2.New_Connections_SA_PV')
                new_conn_mg   = sum([
                    sum_row(t) for t in df.index if '2.New_Connections_MG' in t
                ])
                total = new_conn_grid + new_conn_shs + new_conn_mg

                summary_results.append({
                    'ISO3':             iso3,
                    'NewConn_Grid':     round(new_conn_grid),
                    'NewConn_SHS':      round(new_conn_shs),
                    'NewConn_MG':       round(new_conn_mg),
                    'NewConn_Total':    round(total),
                    'Grid_viable_%':    round(new_conn_grid / total * 100, 1) if total > 0 else 0,
                    'SHS_%':            round(new_conn_shs  / total * 100, 1) if total > 0 else 0,
                    'MiniGrid_%':       round(new_conn_mg   / total * 100, 1) if total > 0 else 0,
                    'OffGrid_viable_%': round((new_conn_shs + new_conn_mg) / total * 100, 1) if total > 0 else 0,
                })
                print(f"{iso3}: done")
    except Exception as e:
        print(f"{iso3}: error — {e}")

gep_summary = pd.DataFrame(summary_results).sort_values('OffGrid_viable_%', ascending=False)
print(gep_summary.to_string(index=False))
print(f"\nSSA totals:")
print(f"  Total new connections: {gep_summary['NewConn_Total'].sum()/1e6:.1f}M")
print(f"  SHS:      {gep_summary['NewConn_SHS'].sum()/1e6:.1f}M ({gep_summary['NewConn_SHS'].sum()/gep_summary['NewConn_Total'].sum()*100:.1f}%)")
print(f"  Grid:     {gep_summary['NewConn_Grid'].sum()/1e6:.1f}M ({gep_summary['NewConn_Grid'].sum()/gep_summary['NewConn_Total'].sum()*100:.1f}%)")
print(f"  MiniGrid: {gep_summary['NewConn_MG'].sum()/1e6:.1f}M ({gep_summary['NewConn_MG'].sum()/gep_summary['NewConn_Total'].sum()*100:.1f}%)")

AGO: done
BEN: done
BWA: done
BFA: done
BDI: done
CMR: done
CAF: done
TCD: done
CPV: HTTP 403
COD: done
COG: done
COM: done
CIV: done
DJI: done
GNQ: done
ERI: done
ETH: done
GAB: done
GMB: done
GHA: done
GIN: done
GNB: done
KEN: done
LSO: done
LBR: done
MDG: done
MWI: done
MLI: done
MRT: done
MOZ: done
MUS: HTTP 403
NAM: done
NER: done
NGA: done
RWA: done
SEN: done
SLE: done
SOM: file not found — available: ['so-2-0_0_0_summary.csv', 'so-2-0_0_1_summary.csv', 'so-2-0_1_0_summary.csv']
STP: done
SYC: HTTP 403
ZAF: done
SSD: done
SDN: done
SWZ: done
TZA: done
TGO: done
UGA: done
ZMB: done
ZWE: done
ISO3  NewConn_Grid  NewConn_SHS  NewConn_MG  NewConn_Total  Grid_viable_%  SHS_%  MiniGrid_%  OffGrid_viable_%
 SSD        364864      4646212    11728370       16739446            2.2   27.8        70.1              97.8
 TCD       1853222      7324170    10360000       19537392            9.5   37.5        53.0              90.5
 CAF        672326      2020808     1865566        4558700     

In [5]:
# Somalia special case — uses 3-parameter scenario format
som_url = 'https://gep-source-archive.s3.amazonaws.com/models-2021/so-2/outputs/so-2-scenarios-summaries.zip'
try:
    r = requests.get(som_url, timeout=30)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        print(f"Somalia available files: {z.namelist()[:5]}")
        # use base scenario 0_0_0
        with z.open('so-2-0_0_0_summary.csv') as f:
            df = pd.read_csv(f, index_col=0)
            year_cols = [c for c in df.columns if c in ['2025', '2030']]
            new_conn_grid = sum_row('2.New_Connections_Grid')
            new_conn_shs  = sum_row('2.New_Connections_SA_PV')
            new_conn_mg   = sum([sum_row(t) for t in df.index if '2.New_Connections_MG' in t])
            total         = new_conn_grid + new_conn_shs + new_conn_mg
            summary_results.append({
                'ISO3': 'SOM',
                'NewConn_Grid':     round(new_conn_grid),
                'NewConn_SHS':      round(new_conn_shs),
                'NewConn_MG':       round(new_conn_mg),
                'NewConn_Total':    round(total),
                'Grid_viable_%':    round(new_conn_grid / total * 100, 1) if total > 0 else 0,
                'SHS_%':            round(new_conn_shs  / total * 100, 1) if total > 0 else 0,
                'MiniGrid_%':       round(new_conn_mg   / total * 100, 1) if total > 0 else 0,
                'OffGrid_viable_%': round((new_conn_shs + new_conn_mg) / total * 100, 1) if total > 0 else 0,
            })
            print(f"SOM: done (3-param scenario)")
except Exception as e:
    print(f"SOM: error — {e}")

# rebuild gep_summary with Somalia
gep_summary = pd.DataFrame(summary_results).sort_values('OffGrid_viable_%', ascending=False)

Somalia available files: ['so-2-0_0_0_summary.csv', 'so-2-0_0_1_summary.csv', 'so-2-0_1_0_summary.csv', 'so-2-0_1_1_summary.csv', 'so-2-1_0_0_summary.csv']
SOM: done (3-param scenario)


In [6]:
gep_summary.to_csv(GEP_SUMMARY_CSV, index=False)
print(f"Saved {len(gep_summary)} countries to {GEP_SUMMARY_CSV}")

Saved 46 countries to c:\Users\USER\Documents\GitHub\SM_Dissertation\GEP Review Code\Pulled Data\gep_summary.csv
